# Defensive Cybersecurity 50M: staged, observable training

This private Kaggle notebook runs each stage in a separate cell so the source audit, pretraining, and SFT progress are visible independently. It fetches licensed defensive sources, omits the optional Cosmopedia download that caused repeated OpenStax file-descriptor retries, derives optimizer steps from the actual prepared chunk counts, and caps each phase at two data passes and 30,000 updates. Validation runs every 500 updates on a fixed sample, with early stopping. This is an experiment, not a claim of broad cybersecurity expertise.


In [ ]:
import glob, os, shutil, subprocess, sys, torch, tokenizers
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle did not provision a GPU; select GPU T4 and retry.')
try:
    import datasets
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'datasets'], check=True)
    import datasets
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
print('PyTorch:', torch.__version__, '| Tokenizers:', tokenizers.__version__, '| Datasets:', datasets.__version__, flush=True)


In [ ]:
# Copy project utilities and the private base corpus from the attached private Kaggle input.
input_root = '/kaggle/input'
print('Kaggle inputs:', os.listdir(input_root), flush=True)
train_scripts = glob.glob(os.path.join(input_root, '**', 'train_gpt.py'), recursive=True)
if not train_scripts:
    raise FileNotFoundError('No attached private input dataset with train_gpt.py found')
bundle = os.path.dirname(train_scripts[0])
work = '/kaggle/working/cyber_50m'
data_dir = '/kaggle/working/data'
os.makedirs(work, exist_ok=True); os.makedirs(data_dir, exist_ok=True)
for name in ['train_gpt.py', 'train_sft.py', 'fetch_additional_sources.py', 'evaluate_tasks.py', 'model_config.json']:
    shutil.copy2(os.path.join(bundle, name), os.path.join(work, name))
for name in ['official_defensive_text.jsonl', 'incident_text.jsonl', 'cybersecurity_qa_train.jsonl', 'holycyber_defensive_train.jsonl', 'cybersecurity_qa_eval.jsonl', 'copilot_style_private.jsonl']:
    source = os.path.join(bundle, name)
    if os.path.exists(source):
        shutil.copy2(source, os.path.join(data_dir, name))
if not os.path.exists(os.path.join(data_dir, 'cybersecurity_qa_eval.jsonl')):
    raise FileNotFoundError('Separate held-out QA evaluation file is missing')
print('Base input copied; source fetch is a separate next cell.', flush=True)


In [ ]:
print('Starting bounded source preparation; Cosmopedia is intentionally skipped.', flush=True)
subprocess.run([sys.executable, '-u', os.path.join(work, 'fetch_additional_sources.py')], check=True, cwd='/kaggle/working', env=env)
print('Source preparation complete.', flush=True)


In [ ]:
pretrain_dir = '/kaggle/working/cyber_50m_pretrain'
print('Starting pretraining: up to two corpus passes, capped at 30,000 optimizer steps.', flush=True)
subprocess.run([sys.executable, '-u', os.path.join(work, 'train_gpt.py'), '--data-dir', data_dir, '--config', os.path.join(work, 'model_config.json'), '--epochs', '2', '--max-steps', '30000', '--batch-size', '2', '--grad-accum', '4', '--eval-interval', '500', '--patience-evals', '6', '--max-eval-chunks', '512', '--log-interval', '50', '--out-dir', pretrain_dir], check=True, cwd='/kaggle/working', env=env)
print('Pretraining complete.', flush=True)


In [ ]:
print('Evaluating pretrained checkpoint on the separate expert-reviewed holdout.', flush=True)
subprocess.run([sys.executable, '-u', os.path.join(work, 'evaluate_tasks.py'), '--checkpoint', os.path.join(pretrain_dir, 'best_checkpoint.pt'), '--tokenizer', os.path.join(pretrain_dir, 'tokenizer.json'), '--eval-file', os.path.join(data_dir, 'cybersecurity_qa_eval.jsonl'), '--output-dir', pretrain_dir, '--max-new-tokens', '96'], check=True, cwd='/kaggle/working', env=env)


In [ ]:
sft_dir = '/kaggle/working/cyber_50m_sft'
print('Starting answer-masked defensive SFT: up to two answer-chunk passes, capped at 30,000 optimizer steps.', flush=True)
subprocess.run([sys.executable, '-u', os.path.join(work, 'train_sft.py'), '--data-dir', data_dir, '--checkpoint', os.path.join(pretrain_dir, 'best_checkpoint.pt'), '--tokenizer', os.path.join(pretrain_dir, 'tokenizer.json'), '--epochs', '2', '--max-steps', '30000', '--batch-size', '2', '--grad-accum', '4', '--learning-rate', '5e-5', '--eval-interval', '500', '--patience-evals', '6', '--max-eval-chunks', '512', '--log-interval', '50', '--out-dir', sft_dir], check=True, cwd='/kaggle/working', env=env)
print('SFT complete.', flush=True)


In [ ]:
print('Evaluating SFT checkpoint on the same untouched expert-reviewed holdout.', flush=True)
subprocess.run([sys.executable, '-u', os.path.join(work, 'evaluate_tasks.py'), '--checkpoint', os.path.join(sft_dir, 'best_sft_checkpoint.pt'), '--tokenizer', os.path.join(pretrain_dir, 'tokenizer.json'), '--eval-file', os.path.join(data_dir, 'cybersecurity_qa_eval.jsonl'), '--output-dir', sft_dir, '--max-new-tokens', '128'], check=True, cwd='/kaggle/working', env=env)


In [ ]:
import json
for label, path in [('source_audit', '/kaggle/working/data/additional_sources_summary.json'), ('pretrain_summary', '/kaggle/working/cyber_50m_pretrain/run_summary.json'), ('pretrain_task_eval', '/kaggle/working/cyber_50m_pretrain/task_evaluation_summary.json'), ('sft_summary', '/kaggle/working/cyber_50m_sft/sft_summary.json'), ('sft_task_eval', '/kaggle/working/cyber_50m_sft/task_evaluation_summary.json')]:
    print('\n' + label + ':\n' + open(path, encoding='utf-8').read(), flush=True)
print('Artifacts:', sorted(os.listdir('/kaggle/working/cyber_50m_sft')), flush=True)
